# HM Land Registry Price Paid Data & the new UPRN look-up · **Silver layer**

**What this notebook does, in one breath:** types the raw CSVs, joins the UPRN
look-up onto Price Paid Data, and — before anything else is allowed to happen —
proves two things: that the join hasn't changed the shape of my fact table, and
that my extraction reproduces a transaction count HM Land Registry published
themselves.

**The decision this notebook is really about:** HMLR's own spec says a single
sale can map to more than one UPRN. If I put a `uprn` column on the fact table,
the first month that happens my row count changes silently on a left join, and
every rate I've ever published from it is quietly wrong. So the UPRNs don't go
on the fact table at all — they go in a bridge.

## Step 0 — Tools I need

`pandas` for the join, `hashlib` to check Bronze hasn't moved under me.

In [40]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

print("Tools loaded OK")

Tools loaded OK


## Step 1 — Point at my folders

In [42]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RELEASE_SLUG = "release_2026-08-28"
BRONZE_DIR = PROJECT_DIR / "data" / "bronze" / RELEASE_SLUG
SILVER_DIR = PROJECT_DIR / "data" / "silver" / RELEASE_SLUG
GOLD_DIR = PROJECT_DIR / "data" / "gold" / RELEASE_SLUG

print("Project:", PROJECT_DIR)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
print("Bronze :", BRONZE_DIR)
print("Silver :", SILVER_DIR)

Project: /Users/yusufismail/hmlr-price-paid-uprn-pipeline
Bronze : /Users/yusufismail/hmlr-price-paid-uprn-pipeline/data/bronze/release_2026-08-28
Silver : /Users/yusufismail/hmlr-price-paid-uprn-pipeline/data/silver/release_2026-08-28


## Step 2 — Has Bronze moved?

Bronze recorded a SHA-256 for each file. Before I build anything on top of it I
check the files still hash to what the manifest says. If they don't, something
has been edited by hand and the honest thing is to stop rather than to analyse
it.

In [44]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = json.loads((BRONZE_DIR / "metadata.json").read_text())

for entry in manifest["files"]:
    actual = sha256(BRONZE_DIR / entry["filename"])
    if actual != entry["sha256"]:
        raise AssertionError(f"{entry['filename']}: Bronze has changed since ingest")
    print(f"{entry['filename']:<40} checksum ok")

pp-monthly-update-new-version.csv        checksum ok
pp-uprn-lookup-jul-2026.csv              checksum ok


## Step 3 — Give the columns their names

Neither file ships a header. The field order comes from HMLR's specification,
and this is the first point in the pipeline where it gets applied.

Two things I'm deliberately *not* doing:

- The postcode stays an empty string where it's absent, not a null and certainly
  not an imputed value. "Incomplete address detail" is one of the spec's own
  reasons a sale won't match, so a missing postcode is signal, not a gap.
- The identifier keeps its braces. All 38 characters, exactly as published.

And one check that stops the pipeline: the four columns the Gold breakdowns
depend on get their domains enforced. If HMLR ever add a property type, I want
to hear about it here rather than see a mystery row in a chart.

In [46]:
PPD_COLUMNS = [
    "tuid", "price", "date_of_transfer", "postcode", "property_type",
    "old_new", "duration", "paon", "saon", "street", "locality",
    "town_city", "district", "county", "ppd_category_type", "record_status",
]

ENFORCED_DOMAINS = {
    "property_type": {"D", "S", "T", "F", "O"},
    "old_new": {"Y", "N"},
    "ppd_category_type": {"A", "B"},
    "record_status": {"A", "C", "D"},
}

ppd = pd.read_csv(BRONZE_DIR / "pp-monthly-update-new-version.csv",
                  header=None, names=PPD_COLUMNS, dtype=str, keep_default_na=False)
ppd["price"] = ppd["price"].astype("int64")
ppd["transfer_date"] = pd.to_datetime(ppd["date_of_transfer"], format="%Y-%m-%d %H:%M")
ppd["transfer_month"] = ppd["transfer_date"].dt.strftime("%Y-%m")
ppd = ppd.drop(columns=["date_of_transfer"])
ppd["has_postcode"] = ppd["postcode"].str.len() > 0

lookup = pd.read_csv(BRONZE_DIR / "pp-uprn-lookup-jul-2026.csv",
                     header=None, names=["tuid", "uprn"], dtype=str, keep_default_na=False)

for column, expected in ENFORCED_DOMAINS.items():
    unexpected = set(ppd[column].unique()) - expected
    if unexpected:
        raise AssertionError(f"{column}: unexpected {sorted(unexpected)}")
    print(f"{column:<20} {sorted(set(ppd[column].unique()))}")

print(f"\nPrice Paid : {len(ppd):,} rows")
print(f"Look-up    : {len(lookup):,} rows")
print(f"Transfer dates run {ppd.transfer_date.min():%Y-%m-%d} to {ppd.transfer_date.max():%Y-%m-%d}")

property_type        ['D', 'F', 'O', 'S', 'T']
old_new              ['N', 'Y']
ppd_category_type    ['A', 'B']
record_status        ['A', 'C', 'D']

Price Paid : 101,600 rows
Look-up    : 94,112 rows
Transfer dates run 1995-01-11 to 2026-07-31


Domains are clean — nothing unexpected in any of the four. Note the transfer
dates: **1995 to 2026 in a file called "July 2026"**. That's the delta behaviour
from Bronze showing up in the data, and it's why the validation gate can only
test the newest month.

## Step 4 — Deletions, and why they leave the denominator

The monthly file carries a Record Status: `A` for an addition, `C` for a change,
`D` for a deletion. It's the one column the complete file doesn't have, and it's
easy to ignore.

Ignoring it would be a mistake here, because the look-up covers live records
only. Let me check what the deleted rows actually do.

In [49]:
ppd["is_matched"] = ppd["tuid"].isin(set(lookup["tuid"]))

status = ppd.groupby("record_status").agg(
    rows=("tuid", "size"), matched=("is_matched", "sum"))
status["unmatched"] = status["rows"] - status["matched"]
status["unmatched_pct"] = (100 * status["unmatched"] / status["rows"]).round(2)
status

,rows,matched,unmatched,unmatched_pct
record_status,,,,
A,96849,91353,5496,5.67
C,3237,2759,478,14.77
D,1514,0,1514,100.00


**Every single deleted record is unmatched — 1,514 of 1,514, 100%.** Not because
HMLR failed to match them, but because they aren't live records any more.

Leaving them in the denominator moves the headline unmatched rate from 5.97% to
7.37%. A 1.4 percentage-point artefact of not reading a column. So they get
flagged rather than dropped — kept in Silver so the decision stays visible and
reversible, excluded in Gold.

In [51]:
ppd["is_analysis_row"] = ppd["record_status"] != "D"

print(f"all rows            : {len(ppd):,}")
print(f"analysis rows       : {ppd.is_analysis_row.sum():,}")
print(f"unmatched, all rows : {100 * (~ppd.is_matched).mean():.2f}%")
print(f"unmatched, analysis : {100 * (~ppd[ppd.is_analysis_row].is_matched).mean():.2f}%")

all rows            : 101,600
analysis rows       : 100,086
unmatched, all rows : 7.37%
unmatched, analysis : 5.97%


## Step 5 — The grain problem, in both directions

HMLR's spec: *"one published sale may be linked to more than one UPRN … because a
sale can cover more than one addressable location, such as several flats,
separate dwellings, or different parts of a property."*

That's the failure mode everyone writing about this dataset will warn about — a
naive left join inflating the fact table. So: how many sales actually map to more
than one UPRN this month?

In [53]:
uprn_count = lookup.groupby("tuid").size().rename("uprn_count")

fct = ppd.merge(uprn_count, left_on="tuid", right_index=True, how="left")
fct["uprn_count"] = fct["uprn_count"].fillna(0).astype("int64")

print("UPRNs per transaction:")
print(fct["uprn_count"].value_counts().sort_index().to_string())
print(f"\nrow count before join : {len(ppd):,}")
print(f"row count after join  : {len(fct):,}")
assert len(fct) == len(ppd), "the join changed the fact table row count"
print("fact table row count preserved")

UPRNs per transaction:
uprn_count
0     7488
1    94112

row count before join : 101,600
row count after join  : 101,600
fact table row count preserved


**None.** Not one transaction maps to more than one UPRN. 94,112 match exactly
one, 7,488 match none, and the left join leaves the row count at 101,600 exactly.

The thing the spec warns about, and the thing I expected to be the headline
caveat, does not occur in the first published month.

But multiplicity is here — it just runs the other way.

In [55]:
per_uprn = lookup.groupby("uprn").size()

print(f"distinct UPRNs                    : {per_uprn.size:,}")
print(f"UPRNs carrying more than one sale : {(per_uprn > 1).sum():,}")
print(f"sales sitting on a repeated UPRN  : {per_uprn[per_uprn > 1].sum():,}")
print(f"most sales on a single UPRN       : {per_uprn.max()}")
print()
print(per_uprn.value_counts().sort_index().rename("UPRNs").to_frame().T.to_string())

distinct UPRNs                    : 92,750
UPRNs carrying more than one sale : 1,055
sales sitting on a repeated UPRN  : 2,417
most sales on a single UPRN       : 7

           1    2    3   4   5  6  7
UPRNs  91695  855  130  45  16  6  3


1,055 UPRNs carry between two and seven sales each. Harmless at transaction
grain — and a double-count the moment anything is grouped by property.

So the model splits in two, and stays split even though this month doesn't need
it:

- **`fct_transaction`** — one row per transaction, carrying `uprn_count` and
  `is_matched`, but *no* `uprn` column.
- **`bridge_transaction_uprn`** — one row per (transaction, UPRN) pair.

That's grain-safe whatever a later month contains. Building it now costs
nothing; retrofitting it after publishing a wrong number costs a lot.

In [57]:
fct["is_matched"] = fct["uprn_count"] > 0
bridge = lookup.copy()

orphans = int((~bridge["tuid"].isin(set(fct["tuid"]))).sum())
assert orphans == 0, f"{orphans} look-up rows reference an unknown transaction"
assert not fct["tuid"].duplicated().any(), "tuid is not unique in the fact table"

print(f"fct_transaction         : {len(fct):,} rows, tuid unique")
print(f"bridge_transaction_uprn : {len(bridge):,} rows, {orphans} orphans")

fct_transaction         : 101,600 rows, tuid unique
bridge_transaction_uprn : 94,112 rows, 0 orphans


## Step 6 — The validation gate

This is the part that decides whether anything downstream is allowed to be
published.

Bronze captured HMLR's own count of transactions dated July 2026 from their
triplestore: 22,835. My extraction has to reproduce it. The three control months
should *not* match, and by a specific amount — the volume that shipped in
earlier releases.

In [59]:
reference = json.loads((BRONZE_DIR / "reconciliation_reference.json").read_text())
target = reference["linked_data"]["target"]

actual = int((fct["transfer_month"] == target["transfer_month"]).sum())
print(f"transfers dated {target['transfer_month']}")
print(f"  my extraction     : {actual:,}")
print(f"  HMLR linked data  : {target['count']:,}")
print(f"  -> {'MATCH' if actual == target['count'] else 'MISMATCH'}\n")

assert actual == target["count"], "validation gate failed — stop here"

rows = []
for control in reference["linked_data"]["controls"]:
    in_release = int((fct["transfer_month"] == control["transfer_month"]).sum())
    rows.append({
        "transfer_month": control["transfer_month"],
        "in_this_release": in_release,
        "hmlr_linked_data_total": control["count"],
        "published_in_earlier_releases": control["count"] - in_release,
    })
pd.DataFrame(rows)

transfers dated 2026-07
  my extraction     : 22,835
  HMLR linked data  : 22,835
  -> MATCH



,transfer_month,in_this_release,hmlr_linked_data_total,published_in_earlier_releases
0,2026-06,27624,46025,18401
1,2026-05,4028,46381,42353
2,2025-09,8303,80340,72037


**22,835 against 22,835.** My extraction reproduces HM Land Registry's own
transaction count exactly, from a completely separate publication channel.

And the controls behave as they must: June shows 27,624 rows in this release
against 46,025 in the triplestore, the missing 18,401 having shipped on 28 July.
That divergence is what turns the July match from a coincidence into evidence —
if all four months had matched, I'd have been measuring something other than
what I thought.

## Step 7 — Write Silver

Everything above is the working. The *artefacts* are written by
`src/pipeline/03_silver_join.py`, which repeats these same steps and is the
only thing that writes `fct_transaction.parquet`, `bridge_transaction_uprn.parquet`
and `validation.json`.

That split is deliberate, and it is a fix. Both this notebook and the script
used to write `validation.json` to the same path, with slightly different key
names in the controls, so whichever I ran last silently won. One writer, one
schema. I run it here and read the report back, so the notebook and the
pipeline cannot tell different stories.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(PROJECT_DIR / "src" / "pipeline" / "03_silver_join.py")],
    capture_output=True, text=True, cwd=PROJECT_DIR,
)
print(result.stdout.strip() or result.stderr.strip())
result.check_returncode()

report = json.loads((SILVER_DIR / "validation.json").read_text())
gate = report["validation_gate"]
print(f"\ngate: {gate['actual']:,} = {gate['expected']:,} -> {'PASS' if gate['passed'] else 'FAIL'}")
for path in sorted(SILVER_DIR.iterdir()):
    print(f"{path.name:<34} {path.stat().st_size:>12,} bytes")

**Silver is done, and the gate is green.**

What Gold inherits: 100,086 analysis rows, a fact table whose grain is asserted
in both directions, and a reconciled transaction count. The bias question is now
allowed to be asked.